# Follow-up study: data efficiency and an augmentation-matched YOLOv8n

Companion to the other notebooks in
[MHKhan50/jersey-number-detection](https://github.com/MHKhan50/jersey-number-detection).
It answers two questions left open by the main results:

| Runs | What it tests |
|---|---|
| JerseyNet (256 px), YOLOv8n from scratch and YOLOv8n from COCO (both 256 px), each on 10% and 25% of the training images | whether pretraining matters more when there is less data |
| YOLOv8n from scratch, 256 px, with JerseyNet's augmentation (no mosaic, no flip) | how much of the gap is augmentation |

Subset runs take the same number of training steps as the full-data runs. Every run is
selected on validation, scored on test with the same evaluator, and also reports AP by number
size. Seeds 0 and 1 finish first; seed 2 runs if time is left.

**Kaggle settings:** Accelerator *GPU T4 x2*, Internet *on*, the `jersey-2` dataset attached
under *Input*. Then *Save Version → Save & Run All (Commit)*. Results go to
`/kaggle/working/study_results`. No new run starts after `TIME_BUDGET_HOURS`, so the session
ends well inside Kaggle's 12-hour limit.


In [ ]:
# Settings
import os

REPO_URL = "https://github.com/MHKhan50/jersey-number-detection.git"
REPO_REF = "main"
TIME_BUDGET_HOURS = 10.0         # no new run starts after this; Kaggle stops at 12 h
SEEDS = [0, 1, 2]
FRACTIONS = [0.1, 0.25]
OUT_DIR = "/kaggle/working/study_results"
INPUT_ROOTS = ["/kaggle/input"]

# Continuous-integration smoke test only: synthetic data, tiny CPU runs.
SMOKE = os.environ.get("JND_SMOKE") == "1"
if SMOKE:
    SEEDS, FRACTIONS, OUT_DIR = [0], [0.5], os.environ.get("JND_OUT", "/tmp/jnd-study-smoke/results")
print({"REPO_REF": REPO_REF, "SEEDS": SEEDS, "FRACTIONS": FRACTIONS, "TIME_BUDGET_HOURS": TIME_BUDGET_HOURS,
       "SMOKE": SMOKE})


In [ ]:
# Environment check and Ultralytics (PyTorch from the Kaggle image is kept as it is)
import subprocess
import sys

import torch

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
elif not SMOKE:
    print("WARNING: no GPU. Set Accelerator to GPU T4 x2.")
try:
    import ultralytics  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.3"], check=True)
import ultralytics

print("ultralytics", ultralytics.__version__)

In [ ]:
# Get the code
import time
from pathlib import Path

REPO_DIR = Path(os.environ.get("JND_REPO_DIR", "/tmp/jersey-number-detection"))
if not (REPO_DIR / "src" / "jersey_detect").exists():
    for attempt in range(3):
        if subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)]).returncode == 0:
            break
        time.sleep(10)
SRC = str(REPO_DIR / "src")
sys.path.insert(0, SRC)
print(subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--format=%H %s"],
                     capture_output=True, text=True).stdout)

In [ ]:
# Find the dataset
from jersey_detect.prepare import find_dataset

if SMOKE:
    from jersey_detect.synthetic import make_dataset
    DATA_ROOT = make_dataset("/tmp/jnd-study-smoke/data")
else:
    DATA_ROOT = find_dataset(INPUT_ROOTS)
if DATA_ROOT is None:
    raise RuntimeError("No dataset: attach jersey-2 under Input (folder with train/, valid/, test/)")
print("Dataset:", DATA_ROOT)

In [ ]:
# Run the study in a separate process (a crash cannot take the notebook down)
env = dict(os.environ, PYTHONPATH=SRC, PYTHONUNBUFFERED="1")
cmd = [sys.executable, "-m", "jersey_detect", "study", "--data", str(DATA_ROOT), "--out", OUT_DIR,
       "--config", str(REPO_DIR / "configs" / "jersey_net.json"), "--hours", str(TIME_BUDGET_HOURS),
       "--seeds", *map(str, SEEDS), "--fractions", *map(str, FRACTIONS)]
if SMOKE:
    cmd.append("--smoke")
print(" ".join(cmd), flush=True)
proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="", flush=True)
print("return code:", proc.wait())


In [ ]:
# Results
import json

summary_path = Path(OUT_DIR) / "summary.json"
if summary_path.exists():
    print((Path(OUT_DIR) / "results.md").read_text())
    summary = json.loads(summary_path.read_text())
    if summary["errors"]:
        print("Errors:", json.dumps(summary["errors"], indent=1))
else:
    print("No summary written; see the output above.")
